# Faster Search via GPU

This notebook runs an optimized first search for MFAs using known lower bounds and upper bounds to optimize the search.

# Search Algorithm

In [9]:
import os
import ast
import math
import itertools
import random as py_random
import pandas as pd
from tqdm import tqdm

import sys
sys.path.insert(0, '..')

from dim_backprop_gpu_only import compute_dimension

In [10]:
# Helper Functions

def calculate_parameter_count(hidden: tuple, d_0: int, d_h: int) -> int:
    """Calculates the total number of weights and biases in the network."""
    sizes = [d_0] + list(hidden) + [d_h]
    return sum(m * n for m, n in zip(sizes[:-1], sizes[1:]))

def is_less_or_equal(t1: tuple, t2: tuple) -> bool:
    """Returns True if every element in t1 is <= the corresponding element in t2."""
    if len(t1) != len(t2):
        return False
    return all(a <= b for a, b in zip(t1, t2))

def evaluate_single_architecture(hidden_tuple: tuple, h: int, d_0: int, d_h: int, exponent: int):
    """Worker function to evaluate a single architecture and format the result."""
    sizes = [d_0] + list(hidden_tuple) + [d_h]
    arch_str = str(sizes)
    
    try:
        _, _, amb, _, dim, _ = compute_dimension(sizes, exponent)
        params = calculate_parameter_count(hidden_tuple, d_0, d_h)
        is_full = (dim == amb)
        
        status = "FULL " if is_full else "SHORT"
        print(f"  [{status}] {arch_str} -> Rank: {dim}/{amb} (Params: {params})")
        
        return {
            "h": h,
            "exponent": exponent,
            "architecture": arch_str,
            "num_parameters": params,
            "dimension_computed": int(dim),
            "ambient_dimension": int(amb),
            "is_full_dimension": is_full,
            "is_minimal": False 
        }
    except Exception as e:
        print(f"  [ERROR] {arch_str} failed: {e}")
        return None

In [11]:
from __future__ import annotations

import ast
import heapq
import math
import os
import pickle
from functools import lru_cache
from pathlib import Path
from typing import Callable, Iterable, Optional, Sequence

import pandas as pd


# ---------------------------------------------------------------------------
# Basic order and dimension utilities
# ---------------------------------------------------------------------------

def coordinatewise_leq(a: Sequence[int], b: Sequence[int]) -> bool:
    """Return True exactly when a_i <= b_i for every coordinate."""
    return len(a) == len(b) and all(x <= y for x, y in zip(a, b))


def homogeneous_dimension(num_variables: int, degree: int) -> int:
    """Dimension of the space of homogeneous degree-`degree` forms."""
    if num_variables < 1 or degree < 0:
        raise ValueError("num_variables must be positive and degree nonnegative")
    return math.comb(num_variables + degree - 1, degree)


def network_ambient_dimension(
    d_0: int, d_L: int, exponent: int, depth: int
) -> int:
    """Ambient affine dimension for a depth-`depth` homogeneous PNN."""
    output_degree = exponent ** max(depth - 1, 0)
    return d_L * homogeneous_dimension(d_0, output_degree)


def parameter_dimension_bound(architecture: Sequence[int]) -> int:
    """
    Parameter-count upper bound after subtracting hidden-neuron scalings:

        sum_i d_{i-1} d_i - sum_{i=1}^{L-1} d_i.
    """
    if len(architecture) < 2:
        raise ValueError("An architecture must contain at least input and output")
    edge_parameters = sum(
        architecture[i] * architecture[i + 1]
        for i in range(len(architecture) - 1)
    )
    hidden_scalings = sum(architecture[1:-1])
    return edge_parameters - hidden_scalings


@lru_cache(maxsize=None)
def ktb_recursive_upper_bound(
    architecture: tuple[int, ...], exponent: int
) -> int:
    """
    Strongest bound obtained recursively from the Kileel--Trager--Bruna cut
    inequality, starting from ambient and parameter-count estimates.

    For a cut at layer k,

        dim V(d_0,...,d_L)
        <= dim V(d_0,...,d_k) + dim V(d_k,...,d_L) - d_k.

    For a quadratic shallow scalar-output segment (a,b,1), the exact
    symmetric determinantal dimension is also used as a leaf estimate.
    """
    if len(architecture) < 2:
        raise ValueError("An architecture must contain at least two widths")

    segment_depth = len(architecture) - 1
    input_width = architecture[0]
    output_width = architecture[-1]
    segment_degree = exponent ** max(segment_depth - 1, 0)

    ambient = output_width * homogeneous_dimension(input_width, segment_degree)
    bound = min(ambient, parameter_dimension_bound(architecture))

    # Exact affine dimension of the quadratic shallow scalar-output variety:
    # symmetric input_width x input_width matrices of rank <= hidden_width.
    if exponent == 2 and segment_depth == 2 and output_width == 1:
        hidden_width = architecture[1]
        rank = min(input_width, hidden_width)
        exact = rank * (2 * input_width - rank + 1) // 2
        bound = min(bound, exact)

    for cut in range(1, len(architecture) - 1):
        left = architecture[: cut + 1]
        right = architecture[cut:]
        cut_bound = (
            ktb_recursive_upper_bound(left, exponent)
            + ktb_recursive_upper_bound(right, exponent)
            - architecture[cut]
        )
        bound = min(bound, cut_bound)

    return max(0, bound)


# ---------------------------------------------------------------------------
# Proven finite upper box and KTB-derived coordinatewise lower bounds
# ---------------------------------------------------------------------------

def derive_hidden_upper_bounds(
    d_0: int, d_L: int, exponent: int, depth: int
) -> tuple[int, ...]:
    """
    Derive a finite, proven upper bound for every hidden width.

    Always used:
        d_i <= dim Sym^{r^i}(k^{d_0}).

    Then iterate the local compression bound
        d_i <= dim Sym^r(k^{d_{i-1}}).

    For r=2, also use simultaneous diagonalization/Waring compression
        d_i <= d_{i-1} d_{i+1}.

    These are necessary conditions for a *minimal* filling architecture.
    """
    if depth < 1:
        raise ValueError("depth must be at least 1")
    if exponent < 1:
        raise ValueError("exponent must be at least 1")

    num_hidden = depth - 1
    if num_hidden == 0:
        return ()

    if exponent == 1:
        # In the linear case the unique MFA has every hidden width equal to
        # min(d_0,d_L), so this is an exact upper box.
        m = min(d_0, d_L)
        return (m,) * num_hidden

    upper = [
        homogeneous_dimension(d_0, exponent ** i)
        for i in range(1, depth)
    ]

    changed = True
    while changed:
        changed = False
        old = upper.copy()
        for index in range(num_hidden):
            previous_cap = d_0 if index == 0 else old[index - 1]
            next_cap = d_L if index == num_hidden - 1 else old[index + 1]

            local_cap = homogeneous_dimension(previous_cap, exponent)
            new_cap = min(old[index], local_cap)

            if exponent == 2:
                new_cap = min(new_cap, previous_cap * next_cap)

            if new_cap < upper[index]:
                upper[index] = new_cap
                changed = True

    return tuple(upper)


def derive_hidden_lower_bounds(
    d_0: int,
    d_L: int,
    exponent: int,
    depth: int,
    upper_bounds: Sequence[int],
) -> tuple[int, ...]:
    """
    Compute coordinatewise necessary lower bounds using the full recursive
    KTB estimate.

    To test whether d_i=q is possible, set every other hidden width to its
    proven upper cap. If even that maximal architecture has recursive KTB
    bound below the target ambient dimension, then no filling architecture
    can have d_i=q.
    """
    num_hidden = depth - 1
    if len(upper_bounds) != num_hidden:
        raise ValueError("upper_bounds has the wrong length")
    if num_hidden == 0:
        return ()

    if exponent == 1:
        m = min(d_0, d_L)
        return (m,) * num_hidden

    ambient = network_ambient_dimension(d_0, d_L, exponent, depth)
    lower = [1] * num_hidden

    # Essential-variable obstruction at the first hidden layer.
    lower[0] = max(lower[0], d_0)

    # Final coefficient matrix has rank at most d_{L-1}.
    final_form_dimension = homogeneous_dimension(
        d_0, exponent ** max(depth - 1, 0)
    )
    lower[-1] = max(lower[-1], min(d_L, final_form_dimension))

    for index in range(num_hidden):
        found = None
        for q in range(lower[index], upper_bounds[index] + 1):
            hidden = list(upper_bounds)
            hidden[index] = q
            architecture = (d_0, *hidden, d_L)
            if ktb_recursive_upper_bound(architecture, exponent) >= ambient:
                found = q
                break
        if found is None:
            raise RuntimeError(
                "The proven upper box contains no KTB-admissible value for "
                f"hidden layer {index + 1}. Check the dimension routines."
            )
        lower[index] = found

    return tuple(lower)


def satisfies_local_mfa_compression(
    hidden: Sequence[int], d_0: int, d_L: int, exponent: int
) -> bool:
    """Check the actual-neighbor compression inequalities for an MFA."""
    widths = (d_0, *hidden, d_L)
    for i in range(1, len(widths) - 1):
        local_cap = homogeneous_dimension(widths[i - 1], exponent)
        if exponent == 2:
            local_cap = min(local_cap, widths[i - 1] * widths[i + 1])
        if widths[i] > local_cap:
            return False
    return True


# ---------------------------------------------------------------------------
# Antichains, database I/O, and checkpointing
# ---------------------------------------------------------------------------

def minimal_antichain(points: Iterable[tuple[int, ...]]) -> set[tuple[int, ...]]:
    """Coordinatewise minimal elements of a finite set."""
    result: set[tuple[int, ...]] = set()
    for point in sorted(set(points), key=lambda x: (sum(x), x)):
        if any(coordinatewise_leq(other, point) for other in result):
            continue
        result = {
            other for other in result if not coordinatewise_leq(point, other)
        }
        result.add(point)
    return result


def maximal_antichain(points: Iterable[tuple[int, ...]]) -> set[tuple[int, ...]]:
    """Coordinatewise maximal elements of a finite set."""
    result: set[tuple[int, ...]] = set()
    for point in sorted(set(points), key=lambda x: (-sum(x), x)):
        if any(coordinatewise_leq(point, other) for other in result):
            continue
        result = {
            other for other in result if not coordinatewise_leq(other, point)
        }
        result.add(point)
    return result


def _atomic_write_csv(df: pd.DataFrame, csv_path: Path) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = csv_path.with_suffix(csv_path.suffix + ".tmp")
    df.to_csv(temporary, index=False)
    os.replace(temporary, csv_path)


def _atomic_write_pickle(data: object, checkpoint_path: Path) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = checkpoint_path.with_suffix(checkpoint_path.suffix + ".tmp")
    with open(temporary, "wb") as handle:
        pickle.dump(data, handle, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temporary, checkpoint_path)


def _parse_hidden(architecture_value: object) -> tuple[int, ...]:
    if isinstance(architecture_value, str):
        architecture = ast.literal_eval(architecture_value)
    else:
        architecture = list(architecture_value)
    return tuple(int(x) for x in architecture[1:-1])


def _bool_value(value: object) -> bool:
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes"}
    return bool(value)


def _recompute_minimality(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    df = df.copy()
    df["is_minimal"] = False

    full_rows = df[df["is_full_dimension"].map(_bool_value)]
    for (h_value, exponent_value), group in full_rows.groupby(["h", "exponent"]):
        parsed = {
            index: tuple(ast.literal_eval(row["architecture"]))
            for index, row in group.iterrows()
        }
        for index, architecture in parsed.items():
            is_minimal = not any(
                other_index != index
                and coordinatewise_leq(other_architecture, architecture)
                and other_architecture != architecture
                for other_index, other_architecture in parsed.items()
            )
            df.at[index, "is_minimal"] = is_minimal

    return df


# ---------------------------------------------------------------------------
# Terminating monotone frontier search
# ---------------------------------------------------------------------------

def parameter_boundary_search(
    d_0: int,
    d_h: int,
    exponent: int,
    L: int,
    *,
    evaluate_architecture: Optional[Callable[..., Optional[dict]]] = None,
    csv_filename: Optional[str] = None,
    checkpoint_filename: Optional[str] = None,
    resume: bool = True,
    checkpoint_every: int = 500,
) -> pd.DataFrame:
    """
    Find all minimal filling architectures for fixed (d_0,d_h,r,L).

    Required user inputs are only d_0, d_h, exponent, and L. The search:

      1. derives a proven finite upper box by layer compression;
      2. derives coordinatewise lower bounds from the full recursive KTB bound;
      3. traverses the finite box upward from that lower corner;
      4. skips Jacobian evaluations when KTB proves nonfilling;
      5. skips points that cannot be MFAs by local compression;
      6. prunes every upper orthant once a filling point is found;
      7. prunes Jacobian calls below known nonfilling points;
      8. writes every new evaluation immediately to the original CSV format;
      9. saves a checkpoint that can be resumed after KeyboardInterrupt.

    The existing project function `evaluate_single_architecture` is used by
    default. It must have signature

        evaluate_single_architecture(hidden, L, d_0, d_h, exponent) -> dict.

    Search completeness is deterministic conditional on the correctness of
    that filling/nonfilling evaluator.
    """
    if min(d_0, d_h, exponent, L) < 1:
        raise ValueError("d_0, d_h, exponent, and L must all be positive")

    if evaluate_architecture is None:
        try:
            evaluate_architecture = globals()["evaluate_single_architecture"]
        except KeyError as exc:
            raise NameError(
                "Pass evaluate_architecture=... or define "
                "evaluate_single_architecture before calling this function."
            ) from exc

    csv_path = Path(
        csv_filename
        if csv_filename is not None
        else f"../data/raw/{d_0}_{d_h}_r{exponent}_architectures.csv"
    )
    checkpoint_path = Path(
        checkpoint_filename
        if checkpoint_filename is not None
        else str(csv_path.with_suffix("")) + f"_L{L}_frontier_checkpoint.pkl"
    )

    required_columns = [
        "h",
        "exponent",
        "architecture",
        "num_parameters",
        "dimension_computed",
        "ambient_dimension",
        "is_full_dimension",
        "is_minimal",
    ]

    if csv_path.exists():
        print(f"Loading existing database from '{csv_path}'...")
        df = pd.read_csv(csv_path)
        for column in required_columns:
            if column not in df.columns:
                df[column] = False if column in {"is_full_dimension", "is_minimal"} else None
    else:
        print("No existing database found. Starting fresh...")
        df = pd.DataFrame(columns=required_columns)

    num_hidden = L - 1
    ambient = network_ambient_dimension(d_0, d_h, exponent, L)
    upper = derive_hidden_upper_bounds(d_0, d_h, exponent, L)
    lower = derive_hidden_lower_bounds(d_0, d_h, exponent, L, upper)

    print(f"\n--- TERMINATING MFA SEARCH: L={L}, r={exponent} ---")
    print(f"Ambient dimension: {ambient}")
    print(f"Derived lower bounds: {lower}")
    print(f"Derived upper bounds: {upper}")
    finite_box_size = math.prod(hi - lo + 1 for lo, hi in zip(lower, upper))
    print(f"Finite lower/upper box size before KTB pruning: {finite_box_size:,}")

    if any(lo > hi for lo, hi in zip(lower, upper)):
        raise RuntimeError("The derived lower and upper boxes are inconsistent")

    # Existing evaluations for this fixed depth and exponent.
    if df.empty:
        current_df = df
    else:
        current_df = df[(df["h"] == L) & (df["exponent"] == exponent)]

    evaluation_status: dict[tuple[int, ...], bool] = {}
    for _, row in current_df.iterrows():
        hidden = _parse_hidden(row["architecture"])
        if len(hidden) == num_hidden:
            evaluation_status[hidden] = _bool_value(row["is_full_dimension"])

    known_full = minimal_antichain(
        point for point, status in evaluation_status.items() if status
    )
    known_short = maximal_antichain(
        point for point, status in evaluation_status.items() if not status
    )

    metadata = {
        "d_0": d_0,
        "d_h": d_h,
        "exponent": exponent,
        "L": L,
        "lower": lower,
        "upper": upper,
        "ambient": ambient,
    }

    def priority(point: tuple[int, ...]) -> tuple[int, int, tuple[int, ...]]:
        return (sum(point), max(point, default=0), point)

    if resume and checkpoint_path.exists():
        with open(checkpoint_path, "rb") as handle:
            state = pickle.load(handle)
        if state.get("metadata") != metadata:
            raise RuntimeError(
                f"Checkpoint '{checkpoint_path}' belongs to different search parameters."
            )
        heap = state["heap"]
        scheduled = state["scheduled"]
        heapq.heapify(heap)
        print(
            f"Resuming checkpoint with {len(heap):,} queued points and "
            f"{len(scheduled):,} scheduled points."
        )
    else:
        start = tuple(lower)
        heap = [(priority(start), start)]
        scheduled = {start}

    def save_checkpoint() -> None:
        _atomic_write_pickle(
            {
                "metadata": metadata,
                "heap": heap,
                "scheduled": scheduled,
            },
            checkpoint_path,
        )

    def add_successors(point: tuple[int, ...]) -> None:
        for index in range(num_hidden):
            if point[index] >= upper[index]:
                continue
            successor = list(point)
            successor[index] += 1
            successor_tuple = tuple(successor)
            if successor_tuple in scheduled:
                continue
            if any(
                coordinatewise_leq(filling_point, successor_tuple)
                for filling_point in known_full
            ):
                continue
            scheduled.add(successor_tuple)
            heapq.heappush(heap, (priority(successor_tuple), successor_tuple))

    def append_result(result: dict, hidden: tuple[int, ...]) -> None:
        nonlocal df
        result = dict(result)
        result.setdefault("h", L)
        result.setdefault("exponent", exponent)
        result.setdefault("architecture", str([d_0, *hidden, d_h]))
        result.setdefault("num_parameters", parameter_dimension_bound((d_0, *hidden, d_h)))
        result.setdefault("ambient_dimension", ambient)
        result.setdefault("is_minimal", False)
        result["architecture"] = str(
            ast.literal_eval(result["architecture"])
            if isinstance(result["architecture"], str)
            else list(result["architecture"])
        )

        df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
        df = df.drop_duplicates(
            subset=["h", "exponent", "architecture"], keep="last"
        )
        _atomic_write_csv(df, csv_path)

    processed_since_checkpoint = 0
    expensive_evaluations = 0
    ktb_pruned = 0
    compression_pruned = 0
    order_pruned = 0
    current: Optional[tuple[int, ...]] = None

    try:
        while heap:
            _, current = heapq.heappop(heap)

            if any(
                coordinatewise_leq(filling_point, current)
                for filling_point in known_full
            ):
                order_pruned += 1
                current = None
                continue

            # A known nonfilling point proves every point below it nonfilling.
            if any(
                coordinatewise_leq(current, short_point)
                for short_point in known_short
            ):
                add_successors(current)
                order_pruned += 1
                current = None
                continue

            # Such a point cannot itself be an MFA, though larger neighbors may.
            if not satisfies_local_mfa_compression(current, d_0, d_h, exponent):
                add_successors(current)
                compression_pruned += 1
                current = None
                continue

            architecture = (d_0, *current, d_h)
            if ktb_recursive_upper_bound(architecture, exponent) < ambient:
                add_successors(current)
                ktb_pruned += 1
                current = None
                continue

            # This point survives every deterministic obstruction and needs the
            # actual Jacobian/dimension computation.
            result = evaluate_architecture(current, L, d_0, d_h, exponent)
            if result is None:
                raise RuntimeError(
                    f"Evaluation returned None for architecture {architecture}. "
                    "The point has been left in the checkpoint for retry."
                )

            expensive_evaluations += 1
            is_full = _bool_value(result["is_full_dimension"])
            evaluation_status[current] = is_full
            append_result(result, current)

            if is_full:
                known_full = minimal_antichain((*known_full, current))
                print(f"  [FILLING] {architecture}")
                # Do not expand: every strict superset is nonminimal.
            else:
                known_short = maximal_antichain((*known_short, current))
                print(f"  [SHORT]   {architecture}")
                add_successors(current)

            current = None
            processed_since_checkpoint += 1
            if processed_since_checkpoint >= checkpoint_every:
                save_checkpoint()
                processed_since_checkpoint = 0
                print(
                    f"  Checkpoint: queue={len(heap):,}, evaluated={expensive_evaluations:,}, "
                    f"KTB-pruned={ktb_pruned:,}."
                )

    except KeyboardInterrupt:
        if current is not None:
            heapq.heappush(heap, (priority(current), current))
        save_checkpoint()
        df = _recompute_minimality(df)
        _atomic_write_csv(df, csv_path)
        print(
            "\n[Interrupt] Search state and CSV were saved. "
            f"Resume by calling the same function again.\nCheckpoint: {checkpoint_path}"
        )
        return df
    except Exception:
        # Preserve the unresolved current point and all completed CSV rows,
        # then propagate the original exception to the caller.
        if current is not None:
            heapq.heappush(heap, (priority(current), current))
        save_checkpoint()
        df = _recompute_minimality(df)
        _atomic_write_csv(df, csv_path)
        raise

    # Exhausting the heap proves that every MFA in the finite box was handled.
    df = _recompute_minimality(df)
    _atomic_write_csv(df, csv_path)
    if checkpoint_path.exists():
        checkpoint_path.unlink()

    print("\nSearch complete.")
    print(f"  Jacobian/dimension evaluations: {expensive_evaluations:,}")
    print(f"  Recursive KTB prunes:           {ktb_pruned:,}")
    print(f"  Local compression prunes:       {compression_pruned:,}")
    print(f"  Order/CSV prunes:               {order_pruned:,}")
    print(f"  Minimal filling antichain:      {sorted(known_full)}")
    print(f"  Database saved to:              {csv_path}")

    return df

# Execute Optimized Search for MFAs

In [12]:
# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------
# The only mathematical inputs are d_0, d_h, exponent, and L.
#
for L in range(2,9):
    df_results = parameter_boundary_search(
        d_0=2,
        d_h=1,
        exponent=2,
        L=L,
    )

Loading existing database from '..\data\raw\2_1_r2_architectures.csv'...

--- TERMINATING MFA SEARCH: L=2, r=2 ---
Ambient dimension: 3
Derived lower bounds: (2,)
Derived upper bounds: (2,)
Finite lower/upper box size before KTB pruning: 1

Search complete.
  Jacobian/dimension evaluations: 0
  Recursive KTB prunes:           0
  Local compression prunes:       0
  Order/CSV prunes:               1
  Minimal filling antichain:      [(2,)]
  Database saved to:              ..\data\raw\2_1_r2_architectures.csv
Loading existing database from '..\data\raw\2_1_r2_architectures.csv'...

--- TERMINATING MFA SEARCH: L=3, r=2 ---
Ambient dimension: 5
Derived lower bounds: (2, 2)
Derived upper bounds: (3, 3)
Finite lower/upper box size before KTB pruning: 4

Search complete.
  Jacobian/dimension evaluations: 0
  Recursive KTB prunes:           0
  Local compression prunes:       0
  Order/CSV prunes:               1
  Minimal filling antichain:      [(2, 2)]
  Database saved to:              ..\

In [ ]:
# Display the minimal architectures found so far
print("\n=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===")
if not df_results.empty:
    minimal_archs = df_results[df_results['is_minimal'] == True]
    
    if not minimal_archs.empty:
        # Sort by parameters for easier reading
        minimal_archs = minimal_archs.sort_values(by="num_parameters")
        print(minimal_archs[["architecture", "num_parameters", "dimension_computed"]].to_string(index=False))
    else:
        print("No minimal full architectures found matching the criteria.")
else:
    print("Database is empty.")


=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===
            architecture  num_parameters  dimension_computed
               [2, 2, 1]               6                   3
            [2, 2, 2, 1]              10                   5
         [2, 3, 3, 2, 1]              23                   9
      [2, 3, 3, 3, 2, 1]              32                  17
   [2, 3, 3, 4, 4, 2, 1]              53                  33
[2, 3, 3, 4, 6, 5, 2, 1]              93                  65
[2, 3, 3, 4, 5, 6, 3, 1]              98                  65
[2, 3, 4, 5, 5, 5, 2, 1]             100                  65
[2, 3, 4, 5, 6, 4, 2, 1]             102                  65
[2, 3, 4, 4, 5, 5, 4, 1]             103                  65
[2, 3, 3, 5, 7, 4, 2, 1]             103                  65
[2, 3, 4, 5, 5, 4, 4, 1]             103                  65
[2, 3, 3, 5, 5, 5, 4, 1]             104                  65
[2, 3, 3, 5, 6, 4, 4, 1]             104                  65
[2, 3, 5, 5, 5, 4, 3, 1]  

# Examining the Data

In [ ]:
# %load is_unimodal.py

import ast

def is_unimodal(data):
    # 1. Parse the string into a list safely
    if isinstance(data, str):
        try:
            # ast.literal_eval safely evaluates strings containing Python literals
            data = ast.literal_eval(data)
        except (ValueError, SyntaxError):
            raise ValueError(f"Could not parse the string: '{data}'. Ensure it is formatted like '[1, 2, 3]'.")
            
    # 2. Validate the data type
    if not isinstance(data, list):
        raise TypeError("Input must be a list or a string representation of a list.")
        
    # 3. Core unimodal logic
    n = len(data)
    if n <= 2:
        return True
        
    i = 0
    
    # Phase 1: Walk up the non-decreasing slope
    while i + 1 < n and data[i] <= data[i + 1]:
        i += 1
        
    # Phase 2: Walk down the non-increasing slope
    while i + 1 < n and data[i] >= data[i + 1]:
        i += 1
        
    # Phase 3: Check if we reached the end
    return i == n - 1

In [ ]:
USER_DEPTH = 7
d0=2
dL=1
r=2

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_full_dimension'] == True)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_full_dimension'] == True)]))

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED POTENTIAL MINIMAL FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True)]))

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED POTENTIAL NONUNIMODAL MINIMAL FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True) & (df['is_unimodal']==False)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True) & (df['is_unimodal']==False)]))

------------------------------DISCOVERED FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
1188,7,2,"[2, 6, 10, 4, 5, 12, 4, 1]",244,65,65,True,False,False
1189,7,2,"[2, 3, 8, 11, 8, 8, 8, 1]",342,65,65,True,False,True
1190,7,2,"[2, 10, 10, 11, 7, 4, 6, 1]",365,65,65,True,False,False
1192,7,2,"[2, 8, 7, 6, 9, 12, 11, 1]",419,65,65,True,False,False
1196,7,2,"[2, 10, 5, 10, 5, 11, 3, 1]",261,65,65,True,False,False
...,...,...,...,...,...,...,...,...,...
4795,7,2,"[2, 5, 3, 4, 5, 6, 3, 1]",108,65,65,True,False,False
4805,7,2,"[2, 4, 3, 4, 5, 6, 3, 1]",103,65,65,True,False,False
4807,7,2,"[2, 3, 5, 5, 5, 4, 3, 1]",106,65,65,True,True,True
4810,7,2,"[2, 3, 3, 4, 5, 6, 3, 1]",98,65,65,True,True,True


927
------------------------------DISCOVERED POTENTIAL MINIMAL FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
3437,7,2,"[2, 3, 3, 4, 6, 5, 2, 1]",93,65,65,True,True,True
4302,7,2,"[2, 3, 3, 5, 6, 4, 4, 1]",104,65,65,True,True,True
4353,7,2,"[2, 3, 4, 5, 5, 5, 2, 1]",100,65,65,True,True,True
4647,7,2,"[2, 3, 3, 6, 6, 4, 3, 1]",108,65,65,True,True,True
4728,7,2,"[2, 3, 3, 5, 5, 5, 4, 1]",104,65,65,True,True,True
4729,7,2,"[2, 3, 4, 5, 4, 6, 4, 1]",110,65,65,True,True,False
4736,7,2,"[2, 3, 4, 5, 5, 4, 4, 1]",103,65,65,True,True,True
4776,7,2,"[2, 3, 4, 5, 6, 4, 2, 1]",102,65,65,True,True,True
4780,7,2,"[2, 3, 4, 4, 5, 5, 4, 1]",103,65,65,True,True,True
4789,7,2,"[2, 3, 3, 5, 7, 4, 2, 1]",103,65,65,True,True,True


13
------------------------------DISCOVERED POTENTIAL NONUNIMODAL MINIMAL FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
4729,7,2,"[2, 3, 4, 5, 4, 6, 4, 1]",110,65,65,True,True,False


1
